In [9]:
import pandas as pd

sheet1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
sheet2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')

df = pd.concat([sheet1, sheet2], ignore_index=True)

print(df.shape)
print(df.columns.tolist())

(1067371, 8)
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [11]:
df.describe(include='all')

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1067371.0,1067371,1062989,1.067371e+06,1067371,1.067371e+06,824364.000000,1067371
unique,53628.0,5305,5698,NaN,NaN,NaN,NaN,43
top,537434.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1350.0,5829,5918,NaN,NaN,NaN,NaN,981330
mean,NaN,NaN,NaN,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504,NaN
min,NaN,NaN,NaN,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000,NaN
50%,NaN,NaN,NaN,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000,NaN
75%,NaN,NaN,NaN,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000,NaN
max,NaN,NaN,NaN,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000,NaN


In [12]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [13]:
df.duplicated().sum()

np.int64(34335)

In [14]:
df[df['Invoice'].astype(str).str.startswith('C')].shape[0]

19494

In [15]:
df[df['Price'] < 0].shape[0]

5

In [16]:
print("Date range:", df['InvoiceDate'].min(), "to", df['InvoiceDate'].max())
print("Unique customers:", df['Customer ID'].nunique())
print("Unique countries:", df['Country'].nunique())
print("Unique products:", df['StockCode'].nunique())
print("Unique invoices:", df['Invoice'].nunique())

Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5942
Unique countries: 43
Unique products: 5305
Unique invoices: 53628


1. Duplicates (34,335 rows) — remove them, not real distinct transactions
2. Missing Customer ID (243,007 rows) — can't be used for customer-level analysis; keep for revenue totals, exclude from customer-level work
3. Cancelled invoices (19,494, prefix "C") — keep but flag, don't delete
4. Negative/zero quantity outside of cancellations — treat as data error, remove
5. Negative price (5 rows) — investigate, likely remove
6. Remove rows where Price = 0, since investigation showed these are stock adjustment entries, not legitimate transactions (e.g., promotional freebies), based on their descriptions and negative quantities.
7. Missing Description (4,382) — fill with "Unknown," don't drop the row
8. Add TotalPrice = Quantity * Price column

In [17]:
df[df['Price'] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [18]:
(df['Price'] == 0).sum()

np.int64(6202)

In [19]:
df[df['Price'] == 0].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom


In [20]:
df_clean = df.copy()
print("Starting rows:", len(df_clean))

df_clean = df_clean.drop_duplicates()
print("After removing duplicates:", len(df_clean))

df_clean['is_cancelled'] = df_clean['Invoice'].astype(str).str.startswith('C')
print("Cancelled rows flagged:", df_clean['is_cancelled'].sum())

df_clean = df_clean[df_clean['Price'] > 0]
print("After removing negative/zero price rows:", len(df_clean))

df_clean = df_clean[~((df_clean['Quantity'] <= 0) & (~df_clean['is_cancelled']))]
print("After removing invalid quantity rows:", len(df_clean))

df_clean['Description'] = df_clean['Description'].fillna('Unknown')
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['Price']
print("Final shape:", df_clean.shape)

Starting rows: 1067371
After removing duplicates: 1033036
Cancelled rows flagged: 19104
After removing negative/zero price rows: 1027017
After removing invalid quantity rows: 1027017
Final shape: (1027017, 10)


In [21]:
df_customer_level = df_clean[df_clean['Customer ID'].notnull()].copy()
print("Full clean dataset:", len(df_clean))
print("Customer-level dataset:", len(df_customer_level))
print("Rows excluded (no Customer ID):", len(df_clean) - len(df_customer_level))

Full clean dataset: 1027017
Customer-level dataset: 797815
Rows excluded (no Customer ID): 229202


In [22]:
df_clean.to_csv('../data/processed/sales_clean_full.csv', index=False)
df_customer_level.to_csv('../data/processed/sales_clean_customer_level.csv', index=False)
print("Saved both files successfully.")

Saved both files successfully.


In [23]:
import os
print("Current working directory:", os.getcwd())
print(os.path.exists('../data/processed/sales_clean_full.csv'))

Current working directory: C:\Users\Hp\OneDrive\Documents\GitHub\project-meridian\notebooks
True


In [24]:
import os
print("Current working directory:", os.getcwd())
print(os.path.exists('../data/processed/sales_clean_full.csv'))

Current working directory: C:\Users\Hp\OneDrive\Documents\GitHub\project-meridian\notebooks
True


In [25]:
df_check = pd.read_csv('../data/processed/sales_clean_full.csv')
print(df_check.shape)
print(df_check['is_cancelled'].sum())
print(df_check.isnull().sum())

(1027017, 10)
19104
Invoice              0
StockCode            0
Description          0
Quantity             0
InvoiceDate          0
Price                0
Customer ID     229202
Country              0
is_cancelled         0
TotalPrice           0
dtype: int64
